In [1]:
import pandas as pd
import numpy as np
import glob
import yfinance as yf

# BLOQUE 1 - Unificación de los 26 CSV

In [2]:
archivos = glob.glob("Datos/Originales/DowJones/*.csv")
dfs = [pd.read_csv(f) for f in archivos]
df_unificado = pd.concat(dfs, ignore_index=True)

empresas = sorted(df_unificado["symbol"].unique())
print(f"Empresas: {len(empresas)}")
print(empresas)
print(df_unificado.shape)

Empresas: 26
['AAPL', 'AMGN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'DOW', 'GS', 'HD', 'IBM', 'INTC', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'V']
(2744418, 9)


# BLOQUE 2 - Primera exploración del dataset unificado

In [3]:
# Tipos de datos y vista general
print(df_unificado.dtypes)
print(df_unificado.head())

open        float64
high        float64
low         float64
last        float64
close       float64
volume      float64
date            str
symbol          str
exchange        str
dtype: object
      open    high      low  last    close  volume                      date  \
0  213.370  213.41  213.300   NaN  213.300     NaN  2018-08-16T21:30:00+0000   
1  213.280  213.41  213.160   NaN  213.300     NaN  2018-08-16T21:00:00+0000   
2  213.430  213.43  212.985   NaN  213.280     NaN  2018-08-16T20:30:00+0000   
3  213.133  213.73  212.985   NaN  213.480     NaN  2018-08-16T20:00:00+0000   
4  213.273  213.30  213.133   NaN  213.133     NaN  2018-08-16T19:30:00+0000   

  symbol exchange  
0   AAPL     IEXG  
1   AAPL     IEXG  
2   AAPL     IEXG  
3   AAPL     IEXG  
4   AAPL     IEXG  


In [4]:
# Nulos por columna
print(df_unificado.isnull().sum())
print((df_unificado.isnull().mean() * 100).round(2))

open             0
high             0
low              0
last        849956
close       611651
volume      847408
date             0
symbol           0
exchange         0
dtype: int64
open         0.00
high         0.00
low          0.00
last        30.97
close       22.29
volume      30.88
date         0.00
symbol       0.00
exchange     0.00
dtype: float64


last tiene un 31% de nulos, close un 22%, volume un 31%. open, high y low no tienen ningún nulo.

In [5]:
# Duplicados
print("Duplicados exactos:", df_unificado.duplicated().sum())
print("Duplicados por symbol+date:", df_unificado.duplicated(subset=["symbol", "date"]).sum())

Duplicados exactos: 0
Duplicados por symbol+date: 0


In [6]:
# Rango de fechas por empresa
df_unificado["date"] = pd.to_datetime(df_unificado["date"])
resumen_fechas = df_unificado.groupby("symbol")["date"].agg(["min", "max", "count"])
print(resumen_fechas)

                             min                       max   count
symbol                                                            
AAPL   2018-06-06 17:30:00+00:00 2021-07-01 00:00:00+00:00   48723
AMGN   2020-05-29 15:35:00+00:00 2021-07-01 00:00:00+00:00   25296
AXP    2020-05-29 15:30:59+00:00 2021-07-01 00:00:00+00:00   95650
BA     2020-05-29 15:35:51+00:00 2021-07-01 00:00:00+00:00  101436
CAT    2018-06-11 17:30:00+00:00 2021-07-01 00:00:00+00:00  122364
CRM    2020-05-29 15:31:30+00:00 2021-07-01 00:00:00+00:00  108823
CSCO   2020-05-29 15:31:26+00:00 2021-07-01 00:00:00+00:00  108585
CVX    2020-05-29 15:31:23+00:00 2021-07-01 00:00:00+00:00  108876
DIS    2020-05-29 15:35:00+00:00 2021-07-01 00:00:00+00:00  109393
DOW    2020-05-29 15:35:56+00:00 2021-07-01 00:00:00+00:00  106437
GS     2018-06-11 17:30:00+00:00 2021-07-01 00:00:00+00:00  133145
HD     2020-05-29 17:55:51+00:00 2021-07-01 00:00:00+00:00  105219
IBM    2020-05-29 17:56:29+00:00 2021-07-01 00:00:00+00:00  10

In [7]:
# Frecuencia temporal: ¿cada cuánto hay una fila?
df_unificado = df_unificado.sort_values(["symbol", "date"])
df_unificado["diff_tiempo"] = df_unificado.groupby("symbol")["date"].diff()
print(df_unificado["diff_tiempo"].value_counts().head(15))

diff_tiempo
0 days 00:01:00    2128282
0 days 00:02:00     363519
0 days 00:05:00     113350
0 days 00:03:00      28256
0 days 00:10:00      24415
0 days 00:15:00      20581
0 days 00:30:00      12084
0 days 00:04:00       8282
0 days 17:30:00       2649
0 days 04:01:00       1885
0 days 13:30:00       1723
0 days 00:06:00       1709
0 days 00:07:00       1065
0 days 05:02:00       1059
0 days 00:00:59        737
Name: count, dtype: int64


La mayoría de las filas están separadas por 1 minuto, con algunos huecos de 2 a 30 minutos, y los saltos grandes corresponden al cierre de un día de mercado y la apertura del siguiente. Esto confirma que los datos son intradía (minuto a minuto), mucho más detallados de lo que necesita el reto, así que habrá que agregarlos a frecuencia diaria.

In [8]:
# Valores imposibles: precios <= 0, volumen negativo
print("Precios <= 0:", (df_unificado[["open","high","low","close"]] <= 0).sum().sum())
print("Volumen negativo:", (df_unificado["volume"] < 0).sum())

Precios <= 0: 0
Volumen negativo: 0


In [9]:
# ¿close está siempre dentro de [low, high]?
inconsistencias = df_unificado[
    (df_unificado["high"] < df_unificado["low"]) |
    (df_unificado["close"] > df_unificado["high"]) |
    (df_unificado["close"] < df_unificado["low"])
]
print("Filas con close fuera de [low, high]:", len(inconsistencias), f"({len(inconsistencias)/len(df_unificado)*100:.1f}%)")

Filas con close fuera de [low, high]: 761432 (27.7%)


Un 27,7% de las filas, 761.432 de 2.744.418, tienen el cierre fuera del rango entre el máximo y el mínimo de su propia barra, lo cual en teoría no debería pasar nunca, ya que el cierre es uno de los precios negociados dentro de ese mismo intervalo. Es un porcentaje demasiado alto para tratarse de errores puntuales de captura, así que antes de decidir cómo tratarlo se investiga un caso concreto en detalle para entender si el problema está en las columnas de máximo y mínimo o en la de cierre.

# BLOQUE 3 - Investigar el 27,7% de inconsistencias: un ejemplo real

Hay un 27,7% de filas donde close no encaja con el rango [low, high]. Eso, en teoría, no debería pasar. ¿Hay que corregir close? ¿hay que corregir high/low? ¿hay que eliminar esas filas?

El objetivo final es entender por qué el 27,7% de filas tienen close fuera de [low, high]. Si se elige un día con mucho movimiento y se observa que close se queda fijo mientras last sí sube, se tiene una prueba clara del motivo del problema: no se trataría de un error aleatorio de captura, sino de que close sistemáticamente no refleja el precio real de la sesión en curso, sino el del día anterior. Esta explicación, comprobada en un caso extremo, se puede generalizar como la causa del 27,7% de inconsistencias detectadas.

In [10]:
# Pasamos las horas a hora de Nueva York, que es donde opera la bolsa
df_unificado["hora_ny"] = df_unificado["date"].dt.tz_convert("America/New_York")
df_unificado["dia_ny"] = df_unificado["hora_ny"].dt.tz_localize(None).dt.normalize()

# Para investigar, elegimos el día con MÁS movimiento de precio de una empresa,
# porque si close está mal calculado, el error se verá muy claro en un día así.
# Usamos CRM como ejemplo, y dejamos que el propio código encuentre el día
# más volátil de su histórico, en vez de elegirlo nosotros a mano.
crm = df_unificado[df_unificado["symbol"] == "CRM"].copy()
crm["rango_dia"] = crm.groupby("dia_ny")["high"].transform("max") - crm.groupby("dia_ny")["low"].transform("min")
dia_mas_movido = crm.loc[crm["rango_dia"].idxmax(), "dia_ny"]
print("Día con más movimiento de precio en CRM:", dia_mas_movido)

# Miramos ese día fila a fila (una de cada 60, para que la tabla sea legible)
ejemplo = crm[crm["dia_ny"] == dia_mas_movido]
print(ejemplo[["hora_ny", "open", "high", "low", "last", "close", "volume"]].iloc[::60])

Día con más movimiento de precio en CRM: 2020-08-26 00:00:00
                         hora_ny    open    high     low     last   close  \
410967 2020-08-26 08:52:00-04:00  249.29  249.29  246.78  246.780  216.05   
410907 2020-08-26 10:32:00-04:00  249.29  276.14  246.78  272.325  216.05   
410847 2020-08-26 11:47:00-04:00  249.29  277.79  246.78  271.130  216.05   
410787 2020-08-26 13:09:00-04:00  249.29  277.79  246.78  274.185  216.05   
410727 2020-08-26 14:25:00-04:00  249.29  277.79  246.78  271.675  216.05   
410667 2020-08-26 15:42:00-04:00  249.29  277.79  246.78  269.630  216.05   

           volume  
410967     4599.0  
410907   580805.0  
410847  1171892.0  
410787  1401530.0  
410727  1596052.0  
410667  1738652.0  


En el día de más movimiento de precio de CRM (26/08/2020), la columna close se mantiene fija en 216,05 durante toda la sesión, mientras que last sube de forma progresiva (de 246,78 a 274,18) siguiendo el movimiento real de la acción, y high también aumenta en consecuencia (de 249,29 a 277,79). Esto confirma que close no refleja el precio de la sesión en curso: se queda congelada en un valor que no cambia por mucho que el precio real suba o baje. Esta es la causa del 27,7% de filas con close fuera del rango [low, high]: al no actualizarse durante el día, en cuanto el precio real se aleja lo suficiente de ese valor fijo, close queda automáticamente fuera del rango de máximo y mínimo de la sesión.

In [11]:
# Comprobamos qué close tenía CRM el día anterior (25/08/2020)
dia_anterior = crm[crm["dia_ny"] == "2020-08-25"]
print(dia_anterior[["hora_ny", "last", "close"]].tail(10))

                         hora_ny     last   close
410977 2020-08-25 15:59:00-04:00  216.100  208.46
410976 2020-08-25 16:05:00-04:00  223.000  208.46
410975 2020-08-25 16:11:00-04:00  232.460  208.46
410974 2020-08-25 16:12:00-04:00  233.640  208.46
410973 2020-08-25 16:16:00-04:00  237.300  208.46
410972 2020-08-25 16:17:00-04:00  236.795  208.46
410971 2020-08-25 16:29:00-04:00  241.400  208.46
410970 2020-08-25 16:36:00-04:00  241.420  208.46
410969 2020-08-25 16:38:00-04:00  240.780  208.46
410968 2020-08-25 20:00:00-04:00  216.050  208.46


El valor fijo que aparece en close durante el día 26/08 (216,05) coincide exactamente con el valor de la columna last del día anterior (25/08), en su última fila registrada, a las 20:00. Esto confirma que close del día 26 no refleja ningún precio negociado ese día: es una copia del último valor de last del día anterior. Se observa además que en el propio día 25/08 se repite el mismo patrón: close se mantiene fijo en un valor distinto (208,46) durante toda la jornada, lo que indica que no es un caso aislado del 26/08, sino un comportamiento sistemático de la columna close en todo el dataset.

Se observa además que el último last registrado el día 25/08 corresponde a las 20:00, varias horas después del cierre habitual de la sesión regular (16:00), con un valor (216,05) distinto al de la última operación dentro de ese horario normal (240,78 a las 16:38). Esto indica actividad fuera de horario (after-hours), habitual en días en que las empresas publican resultados. Por tanto, para calcular el precio de cierre diario a partir de last, habrá que quedarse con el último valor registrado antes de las 16:00, y no con el último del día completo, para evitar incorporar movimientos posteriores al cierre oficial de mercado.

Hasta ahora se ha comprobado el problema de close únicamente en el tramo donde existe la columna last (desde mayo/junio de 2020). Falta comprobar si el tramo anterior (2018 a mayo de 2020, en el que solo hay datos de 6 empresas y no existe la columna last) tiene el mismo problema, o si close funciona correctamente en ese periodo. Se aplica el mismo método que con CRM: buscar, dentro de una de esas empresas (AAPL), el día con mayor movimiento de precio, para que un posible fallo se vea con claridad.

In [12]:
# Comprobamos si close funciona bien en el tramo SIN last (ej: AAPL, 2018-2020)
aapl = df_unificado[
    (df_unificado["symbol"] == "AAPL") &
    (df_unificado["dia_ny"] < "2020-05-01")  # nos aseguramos de estar en el tramo antiguo
].copy()

aapl["rango_dia"] = aapl.groupby("dia_ny")["high"].transform("max") - aapl.groupby("dia_ny")["low"].transform("min")
dia_mas_movido_aapl = aapl.loc[aapl["rango_dia"].idxmax(), "dia_ny"]
print("Día con más movimiento de AAPL (tramo antiguo):", dia_mas_movido_aapl)

ejemplo_aapl = aapl[aapl["dia_ny"] == dia_mas_movido_aapl]
print(ejemplo_aapl[["hora_ny", "open", "high", "low", "last", "close", "volume"]].iloc[::20])

Día con más movimiento de AAPL (tramo antiguo): 2020-03-13 00:00:00
                        hora_ny     open    high      low  last    close  \
17385 2020-03-13 13:30:00-04:00  265.090  265.51  253.490   NaN  259.375   
17365 2020-03-13 15:10:00-04:00  260.605  261.17  258.705   NaN  259.100   
17345 2020-03-13 16:50:00-04:00  261.730  263.00  260.385   NaN  260.400   
17325 2020-03-13 18:30:00-04:00  258.590  260.00  256.235   NaN  259.335   

       volume  
17385     NaN  
17365     NaN  
17345     NaN  
17325     NaN  


En el día de mayor movimiento de AAPL dentro del tramo antiguo (13/03/2020, en plena caída de los mercados por el inicio de la pandemia), la columna close sí varía a lo largo de la sesión (259,375 → 259,100 → 260,400 → 259,335), reflejando con normalidad el movimiento del precio. Se confirma también que last no existe en este tramo (aparece como NaN en todas las filas). Esto demuestra que, en el tramo 2018-2020, close sí es una fuente fiable para calcular el cierre diario, a diferencia de lo que ocurre en el tramo posterior.

El fallo de close se ha comprobado hasta ahora solo en CRM. Para confirmar que no es un caso aislado de esa empresa, sino un comportamiento sistemático de la fuente de datos, se repite la misma comprobación con otra empresa distinta, AAPL, dentro del tramo donde sí existe la columna last (desde mayo/junio de 2020).

In [13]:
# Comprobamos el mismo patrón en otra empresa del tramo CON last (ej: AAPL en 2020)
aapl_2020 = df_unificado[
    (df_unificado["symbol"] == "AAPL") &
    (df_unificado["dia_ny"] >= "2020-05-29")
].copy()

aapl_2020["rango_dia"] = aapl_2020.groupby("dia_ny")["high"].transform("max") - aapl_2020.groupby("dia_ny")["low"].transform("min")
dia_mas_movido_aapl2 = aapl_2020.loc[aapl_2020["rango_dia"].idxmax(), "dia_ny"]
print("Día con más movimiento de AAPL (tramo con last):", dia_mas_movido_aapl2)

ejemplo_aapl2 = aapl_2020[aapl_2020["dia_ny"] == dia_mas_movido_aapl2]
print(ejemplo_aapl2[["hora_ny", "open", "high", "low", "last", "close", "volume"]].iloc[::20])

Día con más movimiento de AAPL (tramo con last): 2020-07-30 00:00:00
                        hora_ny    open     high     low    last   close  \
24053 2020-07-30 09:30:00-04:00  376.83  376.840  376.79  376.84  380.16   
24033 2020-07-30 11:20:00-04:00  376.83  380.125  375.35  379.46  380.16   
24013 2020-07-30 13:25:00-04:00  376.83  383.770  375.35  382.62  380.16   
23993 2020-07-30 15:30:00-04:00  376.83  384.430  375.35  384.15  380.16   

         volume  
24053     200.0  
24033  128606.0  
24013  204360.0  
23993  267823.0  


En el día de mayor movimiento de AAPL dentro de este tramo (30/07/2020), close permanece fija en 380,16 durante toda la sesión, mientras que last sube de forma progresiva (376,84 → 379,46 → 382,62 → 384,15), reflejando el movimiento real del precio. Se confirma así que el fallo detectado en CRM no es un caso aislado, sino un patrón sistemático de la columna close en todo el tramo donde existe last.

CONCLUSION

En conjunto, estas comprobaciones muestran que el dataset tiene dos comportamientos distintos según el tramo temporal: en el periodo 2018 a mayo de 2020 (solo 6 empresas, sin la columna last), close funciona correctamente y refleja el precio real de cada momento. En el periodo desde mayo/junio de 2020 en adelante (todas las empresas, con la columna last disponible), close deja de ser fiable: permanece fija durante toda la sesión mostrando el cierre del día anterior, mientras que last sí refleja el precio real negociado. Por tanto, para calcular el cierre diario correcto en todo el dataset, se usará close en el tramo antiguo (donde no existe last) y last en el tramo posterior, cogiendo en este último caso el último valor registrado antes de las 16:00 para evitar incorporar movimientos de after-hours.

# BLOQUE 4 - Calcular el cierre diario correcto para las 26 empresas

Una vez comprobado que close funciona correctamente en el tramo 2018-2020 (sin last) y que falla sistemáticamente en el tramo desde mayo/junio de 2020 (con last), se calcula el precio de cierre diario para las 26 empresas aplicando esta misma regla en cada caso: si el día tiene datos de last, se usa el último valor registrado antes de las 16:00 (hora de cierre de la sesión regular, evitando así el after-hours); si el día no tiene ningún dato de last, se usa el último close del día. El resto de columnas se calculan de la forma habitual para datos financieros diarios: open como el primer precio del día, high como el máximo, low como el mínimo, y volume como la suma de todo el volumen negociado ese día.

In [14]:
# Nos quedamos solo con los valores de last que están al rededor de las 16:00 (cierre oficial),
# para no mezclar el precio de cierre con movimientos de after-hours
minutos_ny = df_unificado["hora_ny"].dt.hour * 60 + df_unificado["hora_ny"].dt.minute
df_unificado["last_sesion"] = df_unificado["last"].where(minutos_ny <= (16 * 60 + 30))  # hasta las 16:30 (para no cortar a las 16:00 exacto y dejar un  margen)

df_diario = (
    df_unificado
    .sort_values(["symbol", "date"])
    .groupby(["symbol", "dia_ny"], as_index=False)
    .agg(
        open=("open", "first"),
        high=("high", "max"),
        low=("low", "min"),
        last_sesion=("last_sesion", "last"),   # último last antes de las 16:00
        last_dia=("last", "last"),             # último last del día entero (respaldo, si no hay antes de las 16:00)
        close_barra=("close", "last"),         # solo válido si el día no tiene ningún last
        n_last=("last", "count"),              # cuántos valores de last hay ese día (para saber qué tramo es)
        volume=("volume", lambda x: x.max() if x.notna().any() else np.nan),  # Se emplea el valor máximo (max) en lugar de la suma (sum) debido a que el volumen intradía del proveedor es acumulativo a lo largo de la sesión, por lo que el último registro del día refleja el volumen total negociado sin sobreestimarlo artificialmente
    )
    .rename(columns={"dia_ny": "date"})
)

# Regla: si el día tiene last, se usa (preferiblemente antes de las 16:00);
# si no hay ningún last ese día (tramo 2018-2020), se usa close
df_diario["close"] = np.where(
    df_diario["n_last"] > 0,
    df_diario["last_sesion"].fillna(df_diario["last_dia"]),
    df_diario["close_barra"],
)

df_diario = df_diario[["symbol", "date", "open", "high", "low", "close", "volume"]]

print(df_diario.shape)
print(df_diario.head())
print("Días sin cierre calculado:", df_diario["close"].isna().sum())
print("Días sin volumen calculado:", df_diario["volume"].isna().sum())

(10177, 7)
  symbol       date     open     high      low   close  volume
0   AAPL 2018-06-06  193.630  194.000  191.950  194.00     NaN
1   AAPL 2018-06-07  194.155  194.160  192.385  193.50     NaN
2   AAPL 2018-06-08  190.980  191.995  189.805  191.51     NaN
3   AAPL 2018-06-11  191.345  191.970  190.230  191.16     NaN
4   AAPL 2018-06-12  191.385  192.600  191.175  192.32     NaN
Días sin cierre calculado: 94
Días sin volumen calculado: 3272


Hay 94 filas (de las 10.177 totales) donde, después de aplicar toda la lógica, close quedó en NaN, es decir, 94 combinaciones de empresa+día donde no había ni last disponible ni close disponible para calcular el cierre.

Además, hay 3.272 filas donde el volumen queda en NaN, ya que ese día no existía ningún dato de volumen

In [15]:
#Volver a mirar el ejemplo de CRM (25/08 y 26/08) para ver si ha hecho bien la transformacion
crm_diario = df_diario[df_diario["symbol"] == "CRM"].sort_values("date")
print(crm_diario[(crm_diario["date"] >= "2020-08-24") & (crm_diario["date"] <= "2020-08-27")])

     symbol       date     open    high     low   close      volume
2456    CRM 2020-08-24  209.105  209.29  205.15  208.50   5093688.0
2457    CRM 2020-08-25  213.500  241.42  210.80  241.40  23880535.0
2458    CRM 2020-08-26  249.290  277.97  246.78  273.41  62557558.0
2459    CRM 2020-08-27  267.600  278.28  260.84  278.13  22710092.0


Con el margen ampliado hasta las 16:30, los cierres diarios de CRM alrededor del día de resultados (24 a 27 de agosto de 2020) reflejan correctamente la subida real de la acción, en lugar de los valores fijos e incorrectos que daba la columna close original. El cálculo queda validado para este ejemplo.

Para Volume: coger un día y una empresa, y ver si el volumen sube de forma constante a lo largo del día (acumulativo) o si son valores independientes por minuto.

In [16]:
# Comprobamos si el volumen es acumulativo dentro de un día, usando CRM el 26/08/2020
crm_vol = df_unificado[(df_unificado["symbol"] == "CRM") & (df_unificado["dia_ny"] == "2020-08-26")]
print(crm_vol[["hora_ny", "volume"]].dropna().iloc[::30])

                         hora_ny     volume
410967 2020-08-26 08:52:00-04:00     4599.0
410937 2020-08-26 09:59:00-04:00   322712.0
410907 2020-08-26 10:32:00-04:00   580805.0
410877 2020-08-26 11:09:00-04:00   883533.0
410847 2020-08-26 11:47:00-04:00  1171892.0
410817 2020-08-26 12:27:00-04:00  1262946.0
410787 2020-08-26 13:09:00-04:00  1401530.0
410757 2020-08-26 13:46:00-04:00  1523294.0
410727 2020-08-26 14:25:00-04:00  1596052.0
410697 2020-08-26 15:04:00-04:00  1672768.0
410667 2020-08-26 15:42:00-04:00  1738652.0


Se comprueba, con CRM el 26/08/2020, que la columna volume es acumulativa a lo largo del día: los valores aumentan de forma constante desde la apertura (4.599 a las 08:52) hasta el final de la sesión (1.738.652 a las 15:42), sin descender en ningún momento. Esto confirma que usar sum() para agregar el volumen diario (como se hizo inicialmente en el Bloque 4) sería un error grave, ya que sumaría el acumulado varias veces, inflando artificialmente el resultado. La agregación correcta es max() (o, de forma equivalente, el último valor del día), que recoge el total acumulado real sin duplicarlo.

# Bloque 5 - Descargar Yahoo Finance

Hasta ahora se ha trabajado únicamente con el CSV proporcionado, que cubre el periodo de 2018 (o 2020, según la empresa) hasta julio de 2021. El objetivo del reto requiere trabajar con la ventana 2016-2025, así que es necesario completar el dataset con una fuente externa. En este bloque se descarga de Yahoo Finance el histórico diario de las mismas 26 empresas del CSV, para todo el rango 2016-2025. Junto con los precios, se descargan también los eventos de split de cada empresa, que se usarán en el siguiente bloque para comprobar si los precios de ambas fuentes son directamente comparables.

In [17]:
FECHA_INICIO = "2016-01-01"
FECHA_FIN = "2026-01-01"   # exclusivo en yfinance -> el último día real incluido es 2025-12-31

partes, splits, fallidos = [], {}, []

for ticker in empresas:
    try:
        activo = yf.Ticker(ticker)
        hist = activo.history(start=FECHA_INICIO, end=FECHA_FIN, auto_adjust=False)
        if hist.empty:
            fallidos.append(ticker)
            continue

        hist = hist.reset_index()[["Date", "Open", "High", "Low", "Close", "Volume"]]
        hist.columns = ["date", "open", "high", "low", "close", "volume"]
        hist["date"] = pd.to_datetime(hist["date"]).dt.tz_localize(None).dt.normalize()
        hist["symbol"] = ticker
        partes.append(hist)

        # Guardamos también los splits de cada empresa, se usarán en el siguiente bloque
        eventos = activo.splits
        eventos.index = eventos.index.tz_localize(None).normalize()
        splits[ticker] = eventos[(eventos.index >= FECHA_INICIO) & (eventos.index < FECHA_FIN)]

    except Exception as error:
        fallidos.append(ticker)
        print(f"{ticker}: falló la descarga -> {error}")

df_yahoo = pd.concat(partes, ignore_index=True)

print("Descargadas:", df_yahoo["symbol"].nunique(), "de", len(empresas))
print("Fallidas:", fallidos)
print(df_yahoo.shape)
print(df_yahoo.groupby("symbol")["date"].agg(["min", "max", "count"]))

Descargadas: 26 de 26
Fallidas: []
(64557, 7)
              min        max  count
symbol                             
AAPL   2016-01-04 2025-12-31   2514
AMGN   2016-01-04 2025-12-31   2514
AXP    2016-01-04 2025-12-31   2514
BA     2016-01-04 2025-12-31   2514
CAT    2016-01-04 2025-12-31   2514
CRM    2016-01-04 2025-12-31   2514
CSCO   2016-01-04 2025-12-31   2514
CVX    2016-01-04 2025-12-31   2514
DIS    2016-01-04 2025-12-31   2514
DOW    2019-03-20 2025-12-31   1707
GS     2016-01-04 2025-12-31   2514
HD     2016-01-04 2025-12-31   2514
IBM    2016-01-04 2025-12-31   2514
INTC   2016-01-04 2025-12-31   2514
JNJ    2016-01-04 2025-12-31   2514
JPM    2016-01-04 2025-12-31   2514
KO     2016-01-04 2025-12-31   2514
MCD    2016-01-04 2025-12-31   2514
MMM    2016-01-04 2025-12-31   2514
MRK    2016-01-04 2025-12-31   2514
MSFT   2016-01-04 2025-12-31   2514
NKE    2016-01-04 2025-12-31   2514
PG     2016-01-04 2025-12-31   2514
TRV    2016-01-04 2025-12-31   2514
UNH    2016-01-04 

Las 26 empresas se descargan correctamente, sin ninguna fallida. Todas cubren el rango completo 2016-2025 salvo DOW, que solo tiene datos desde el 20/03/2019, fecha en la que la empresa empezó a cotizar tras escindirse de DowDuPont. No es un error de la descarga: antes de esa fecha, DOW no existía como empresa cotizada en ninguna fuente.

# BLOQUE 6 - Comprobar y aplicar el ajuste por splits

Antes de combinar el CSV con Yahoo Finance, es necesario resolver dos problemas de escala que impedirían tratar ambas fuentes como comparables: los splits (divisiones de acciones) y un cambio detectado en cómo el CSV reporta el volumen a partir de marzo de 2021.

Antes de ajustar el CSV, se comprueba si el close de Yahoo Finance ya viene corregido por splits, comparándolo con el close del CSV sin ajustar, para el caso de AAPL (que tuvo un split 4:1 el 31/08/2020).

In [ ]:
def obtener_factor_ajuste(ticker, fecha, diccionario_splits):
    """
    Calcula el producto de los factores de split posteriores a una fecha dada.
    Si no hay splits posteriores, devuelve 1.0 (sin cambios en la escala).
    """
    eventos = diccionario_splits.get(ticker)
    if eventos is None or eventos.empty:
        return 1.0
    futuros = eventos[eventos.index > fecha]
    if futuros.empty:
        return 1.0
    return float(np.prod(futuros.values))

comparacion = df_diario.merge(
    df_yahoo[["symbol", "date", "close"]].rename(columns={"close": "close_yahoo"}),
    on=["symbol", "date"], how="inner"
).dropna(subset=["close"])

comparacion["factor_esperado"] = [
    obtener_factor_ajuste(s, f, splits) for s, f in zip(comparacion["symbol"], comparacion["date"])
]
comparacion["cociente"] = comparacion["close"] / comparacion["close_yahoo"]

print("--- Comprobación: ¿Yahoo ya viene ajustado por splits? (ejemplo AAPL) ---")
print(comparacion[comparacion["symbol"] == "AAPL"].groupby("factor_esperado")["cociente"].median())

--- Comprobación: ¿Yahoo ya viene ajustado por splits? (ejemplo AAPL) ---
factor_esperado
1.0    1.000057
4.0    4.000000
Name: cociente, dtype: float64


El cociente entre el close del CSV (sin ajustar) y el de Yahoo coincide exactamente con el factor de split esperado: 1,0 después del split y 4,0 antes del split. Esto confirma que el close de Yahoo Finance ya viene ajustado por splits, y que es necesario aplicar el mismo ajuste al CSV para que ambas fuentes queden en la misma escala antes de combinarlas.

In [ ]:
#Splits: aplicar el ajuste
df_diario["factor_split"] = [
    obtener_factor_ajuste(row["symbol"], row["date"], splits)
    for _, row in df_diario.iterrows()
]

for col in ["open", "high", "low", "close"]:
    df_diario[col] = df_diario[col] / df_diario["factor_split"]

df_diario["volume"] = df_diario["volume"] * df_diario["factor_split"]

df_diario = df_diario.drop(columns=["factor_split"])

In [20]:
#Comprobación final, con AAPL alrededor de su split (31/08/2020) 
comprobacion_aapl = df_diario[
    (df_diario["symbol"] == "AAPL") &
    (df_diario["date"] >= "2020-08-25") &
    (df_diario["date"] <= "2020-09-04")
].sort_values("date")

print("\n--- Comprobación de precios ajustados en AAPL (entorno del split 31/08/2020) ---")
print(comprobacion_aapl[["date", "open", "high", "low", "close", "volume"]])


--- Comprobación de precios ajustados en AAPL (entorno del split 31/08/2020) ---
          date       open      high        low      close       volume
577 2020-08-25  124.51875  125.1793  123.05250  124.72000  208941716.0
578 2020-08-26  126.15250  126.9925  125.08250  126.45875  161004836.0
579 2020-08-27  127.26250  127.4850  123.83250  125.19250  154146696.0
580 2020-08-28  126.05750  126.4375  124.58625  124.85750    1611860.0
581 2020-08-31  125.28000  131.0000  125.07000  129.43000    2563118.0
582 2020-09-01  132.19000  134.7800  130.56000  133.79000    1773719.0
583 2020-09-02  137.84000  138.4200  127.00000  131.69000  198233370.0
584 2020-09-03  128.11000  128.7850  120.18000  120.24000    2675201.0
585 2020-09-04  118.24000  123.7000  110.89000  120.37000  326926347.0


In [23]:
print(df_diario[(df_diario["symbol"] == "AAPL") & (df_diario["date"] == "2020-08-27")][["date", "open", "high", "low", "close"]])
print(df_yahoo[(df_yahoo["symbol"] == "AAPL") & (df_yahoo["date"] == "2020-08-27")][["date", "open", "high", "low", "close"]])

          date      open     high       low     close
579 2020-08-27  127.2625  127.485  123.8325  125.1925
           date        open        high         low       close
1171 2020-08-27  127.142502  127.485001  123.832497  125.010002


Tras aplicar el ajuste, el close de AAPL el 27/08/2020 (125,19) coincide prácticamente con el de Yahoo Finance para esa misma fecha (125,01), con una diferencia mínima atribuible al ruido normal entre fuentes. Esto confirma que el ajuste de splits se ha aplicado correctamente.

Se compara el volumen del CSV contra el de Yahoo Finance, mes a mes, para todas las empresas.

In [21]:
vol_check = df_diario.merge(
    df_yahoo[["symbol", "date", "volume"]].rename(columns={"volume": "volume_yahoo"}),
    on=["symbol", "date"], how="inner"
)
vol_check = vol_check.dropna(subset=["volume", "volume_yahoo"])
vol_check["cociente_vol"] = vol_check["volume"] / vol_check["volume_yahoo"]

# Agrupamos por mes para ver si el problema se concentra en un periodo concreto
print(vol_check.groupby(vol_check["date"].dt.to_period("M"))["cociente_vol"].agg(["median", "min"]).round(3))

         median    min
date                  
2020-05   1.000  1.000
2020-06   0.995  0.001
2020-07   1.000  0.010
2020-08   0.999  0.008
2020-09   1.000  0.005
2020-10   0.998  0.011
2020-11   1.000  0.017
2020-12   1.000  0.009
2021-01   1.000  0.023
2021-02   1.000  0.012
2021-03   0.039  0.000
2021-04   0.034  0.008
2021-05   0.033  0.010
2021-06   0.032  0.011


Se observa un cambio claro de comportamiento: entre mayo de 2020 y febrero de 2021, la mediana mensual del cociente es prácticamente 1 (el CSV coincide con el volumen consolidado de Yahoo), salvo algunos días sueltos ya identificados como fallos puntuales del proveedor. A partir de marzo de 2021, la mediana mensual cae de forma sistemática a un 3-4%, de forma consistente en todas las empresas, mes tras mes. Esto indica que, desde esa fecha, el CSV deja de reportar el volumen consolidado de todo el mercado y pasa a dar solo el volumen negociado en una bolsa concreta (probablemente IEX). No es un valor faltante ni un error puntual: es un cambio de qué mide la columna a partir de esa fecha.

# BLOQUE 7 - Fusión definitiva de CSV y Yahoo Finance (Dataset 2016-2025)

Antes de combinar nada, necesitamos saber qué días son realmente días de mercado (para no arrastrar festivos que pudieran colarse en el CSV). Yahoo Finance, al ser una fuente que solo da datos de días en que el mercado abrió, nos sirve como "calendario de referencia".

In [25]:
#Calendario de días de mercado, usando Yahoo como referencia
#(Yahoo solo tiene filas para días en que el mercado realmente abrió)

calendario = df_yahoo[["symbol", "date"]].drop_duplicates()
print("Días de mercado por empresa (según Yahoo):")
print(calendario.groupby("symbol")["date"].count())

Días de mercado por empresa (según Yahoo):
symbol
AAPL    2514
AMGN    2514
AXP     2514
BA      2514
CAT     2514
CRM     2514
CSCO    2514
CVX     2514
DIS     2514
DOW     1707
GS      2514
HD      2514
IBM     2514
INTC    2514
JNJ     2514
JPM     2514
KO      2514
MCD     2514
MMM     2514
MRK     2514
MSFT    2514
NKE     2514
PG      2514
TRV     2514
UNH     2514
V       2514
Name: date, dtype: int64


In [26]:
#Rellenar los valores sueltos que faltan dentro del propio CSV
# (close o volume en NaN, para días donde el CSV sí tiene fila)

df_diario_relleno = df_diario.merge(
    df_yahoo[["symbol", "date", "close", "volume"]].rename(
        columns={"close": "close_yahoo", "volume": "volume_yahoo"}
    ),
    on=["symbol", "date"], how="left"
)

# Antes de rellenar, contamos cuántos huecos hay
print("Cierres sin dato antes de rellenar:", df_diario_relleno["close"].isna().sum())
print("Volúmenes sin dato antes de rellenar:", df_diario_relleno["volume"].isna().sum())

df_diario_relleno["close"] = df_diario_relleno["close"].fillna(df_diario_relleno["close_yahoo"])
df_diario_relleno["volume"] = df_diario_relleno["volume"].fillna(df_diario_relleno["volume_yahoo"])

print("Cierres sin dato después de rellenar:", df_diario_relleno["close"].isna().sum())
print("Volúmenes sin dato después de rellenar:", df_diario_relleno["volume"].isna().sum())

df_diario_relleno = df_diario_relleno.drop(columns=["close_yahoo", "volume_yahoo"])

Cierres sin dato antes de rellenar: 94
Volúmenes sin dato antes de rellenar: 5485
Cierres sin dato después de rellenar: 0
Volúmenes sin dato después de rellenar: 114


In [27]:
# Miramos qué queda sin volumen, incluso después de intentar rellenar con Yahoo
sin_volumen = df_diario_relleno[df_diario_relleno["volume"].isna()]
print("Fechas de los 114 casos:")
print(sin_volumen["date"].dt.strftime("%Y-%m-%d %a").value_counts().head(15))
print()
print("Empresas afectadas:")
print(sin_volumen["symbol"].value_counts())

Fechas de los 114 casos:
date
2018-07-04 Wed    6
2018-09-03 Mon    6
2018-11-22 Thu    6
2018-12-05 Wed    6
2018-12-25 Tue    6
2019-01-01 Tue    6
2019-01-21 Mon    6
2019-02-18 Mon    6
2019-04-19 Fri    6
2019-05-27 Mon    6
2019-07-04 Thu    6
2019-09-02 Mon    6
2019-11-28 Thu    6
2019-12-25 Wed    6
2020-01-01 Wed    6
Name: count, dtype: int64

Empresas afectadas:
symbol
AAPL    19
CAT     19
GS      19
INTC    19
NKE     19
V       19
Name: count, dtype: int64


Tras el relleno de huecos con Yahoo Finance, quedan 114 filas sin volumen, correspondientes en su totalidad a las 6 empresas del tramo antiguo (AAPL, CAT, GS, INTC, NKE y V), con 19 días cada una. Al revisar las fechas, se confirma que todas coinciden con festivos oficiales de la bolsa de EE. UU. (4 de julio, Acción de Gracias, Navidad, Año Nuevo...), días en que el mercado no abre. El CSV trae filas para estos días (probablemente repitiendo el último precio conocido), pero Yahoo Finance, al no considerarlos días de mercado, no tiene ningún dato con el que rellenarlos. Se eliminan estas 114 filas, ya que representan días de mercado inexistentes: mantenerlas introduciría retornos de 0% falsos que distorsionarían el cálculo de la volatilidad.

In [ ]:
# Eliminamos las 114 filas correspondientes a festivos (sin volumen tras el intento de relleno)
print("Filas antes de eliminar festivos:", len(df_diario_relleno))

df_diario_relleno = df_diario_relleno.dropna(subset=["volume"]).reset_index(drop=True)

print("Filas después de eliminar festivos:", len(df_diario_relleno))

Filas antes de eliminar festivos: 10177
Filas después de eliminar festivos: 10063


In [ ]:
#Añadir los días/años que el CSV no cubre en absoluto (vienen enteros de Yahoo)

# Identificamos qué combinaciones empresa+día YA tenemos (del CSV, ya limpio)
ya_tenemos = set(zip(df_diario_relleno["symbol"], df_diario_relleno["date"]))

# De Yahoo, nos quedamos solo con los días que NO están ya cubiertos por el CSV
df_yahoo_extra = df_yahoo[
    ~df_yahoo.apply(lambda row: (row["symbol"], row["date"]) in ya_tenemos, axis=1)
].copy()

print("Filas que vienen enteras de Yahoo (fuera del rango del CSV):", len(df_yahoo_extra))
print(df_yahoo_extra.groupby("symbol")["date"].agg(["min", "max", "count"]))

Filas que vienen enteras de Yahoo (fuera del rango del CSV): 54494
              min        max  count
symbol                             
AAPL   2016-01-04 2025-12-31   1742
AMGN   2016-01-04 2025-12-31   2241
AXP    2016-01-04 2025-12-31   2241
BA     2016-01-04 2025-12-31   2241
CAT    2016-01-04 2025-12-31   1745
CRM    2016-01-04 2025-12-31   2241
CSCO   2016-01-04 2025-12-31   2241
CVX    2016-01-04 2025-12-31   2241
DIS    2016-01-04 2025-12-31   2241
DOW    2019-03-20 2025-12-31   1434
GS     2016-01-04 2025-12-31   1744
HD     2016-01-04 2025-12-31   2243
IBM    2016-01-04 2025-12-31   2243
INTC   2016-01-04 2025-12-31   1743
JNJ    2016-01-04 2025-12-31   2241
JPM    2016-01-04 2025-12-31   2241
KO     2016-01-04 2025-12-31   2241
MCD    2016-01-04 2025-12-31   2241
MMM    2016-01-04 2025-12-31   2243
MRK    2016-01-04 2025-12-31   2243
MSFT   2016-01-04 2025-12-31   2243
NKE    2016-01-04 2025-12-31   1745
PG     2016-01-04 2025-12-31   2242
TRV    2016-01-04 2025-12-31   22

In [35]:
columnas_finales = ["symbol", "date", "open", "high", "low", "close", "volume"]

df_master = pd.concat([
    df_diario_relleno[columnas_finales],
    df_yahoo_extra[columnas_finales],
], ignore_index=True)

df_master = df_master.sort_values(["symbol", "date"]).reset_index(drop=True)

print(df_master.shape)
print(df_master.groupby("symbol")["date"].agg(["min", "max", "count"]))
print()
print("Nulos por columna:")
print(df_master.isna().sum())

(64557, 7)
              min        max  count
symbol                             
AAPL   2016-01-04 2025-12-31   2514
AMGN   2016-01-04 2025-12-31   2514
AXP    2016-01-04 2025-12-31   2514
BA     2016-01-04 2025-12-31   2514
CAT    2016-01-04 2025-12-31   2514
CRM    2016-01-04 2025-12-31   2514
CSCO   2016-01-04 2025-12-31   2514
CVX    2016-01-04 2025-12-31   2514
DIS    2016-01-04 2025-12-31   2514
DOW    2019-03-20 2025-12-31   1707
GS     2016-01-04 2025-12-31   2514
HD     2016-01-04 2025-12-31   2514
IBM    2016-01-04 2025-12-31   2514
INTC   2016-01-04 2025-12-31   2514
JNJ    2016-01-04 2025-12-31   2514
JPM    2016-01-04 2025-12-31   2514
KO     2016-01-04 2025-12-31   2514
MCD    2016-01-04 2025-12-31   2514
MMM    2016-01-04 2025-12-31   2514
MRK    2016-01-04 2025-12-31   2514
MSFT   2016-01-04 2025-12-31   2514
NKE    2016-01-04 2025-12-31   2514
PG     2016-01-04 2025-12-31   2514
TRV    2016-01-04 2025-12-31   2514
UNH    2016-01-04 2025-12-31   2514
V      2016-01-04

Se combinan ambas fuentes en un dataset maestro único, siguiendo el criterio de que el CSV manda siempre que tenga dato y Yahoo Finance rellena lo que falta. El resultado final cubre las 26 empresas para todo el rango 2016-2025 (2019-2025 en el caso de DOW, que no cotizaba antes), sin ningún valor nulo en ninguna columna, tras haber resuelto en los pasos anteriores los huecos de cierre, el problema del volumen desde marzo de 2021, y los 114 días correspondientes a festivos de mercado.

In [ ]:
#df_master.to_csv("Datos/Transformados/dowjones_diario_2016_2025.csv", index=False)
#print("Guardado:", df_master.shape)